In [1]:
import pandas as pd
import numpy as np

# Load raw dataset
df = pd.read_csv('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Inspect the first 5 records and check structural info
print(df.head())
print(df.info())

   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies        Contract Pape

In [2]:
# Convert TotalCharges to numeric, turning spaces into NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Check how many missing values exist
print(f"Missing values in TotalCharges: {df['TotalCharges'].isnull().sum()}")

# Tenure for these missing records is 0 (brand new customers who haven't paid yet)
# Fill missing TotalCharges with 0
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# Check for duplicate customer IDs
duplicates = df['customerID'].duplicated().sum()
print(f"Duplicate customer records: {duplicates}")


Missing values in TotalCharges: 11
Duplicate customer records: 0


In [3]:
# 1. Tenure Buckets (helps identify WHEN customers churn)
bins = [-1, 12, 24, 48, 72]
labels = ['0-12 Months', '13-24 Months', '25-48 Months', '49+ Months']
df['tenure_cohort'] = pd.cut(df['tenure'], bins=bins, labels=labels)

# 2. Binary Churn Flag (1 for Yes, 0 for No - useful for mathematical aggregates)
df['churn_numeric'] = df['Churn'].apply(lambda x: 1 if x == 'Yes' else 0)

# 3. High Risk Flag (Month-to-month contracts without support services)
df['is_high_risk'] = np.where(
    (df['Contract'] == 'Month-to-month') & 
    (df['OnlineSecurity'] == 'No') & 
    (df['TechSupport'] == 'No'), 
    1, 0
)

# Export cleaned file for SQL and Power BI
df.to_csv('../data/processed/cleaned_churn_data.csv', index=False)
print("Data saved successfully to data/processed/cleaned_churn_data.csv")


Data saved successfully to data/processed/cleaned_churn_data.csv
